In [ ]:
import pandas as pd
import re
import openai
import numpy as np
import pickle
import random
import tabulate
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split, GridSearchCV
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score, 
                             confusion_matrix, accuracy_score, precision_score, recall_score, f1_score)    

In [3]:
# === Utilities ===
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    return text

def compute_proportional_ranks(ranks):
    rank_to_weight = {1: 4, 2: 3, 3: 2, 4: 1}
    total_weight = sum(rank_to_weight[r] for r in ranks)
    return [rank_to_weight[r] / total_weight for r in ranks]

def get_ada_embedding(text, engine="text-embedding-ada-002"):
    response = openai.Embedding.create(input=text, engine=engine)
    return response["data"][0]["embedding"]

def convert_to_ranking(scores):
    sorted_models = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return {model: rank + 1 for rank, (model, _) in enumerate(sorted_models)}


In [4]:
# === Category Mapping ===
category_map = {
    'Summaries': ['summaries'],
    'English': ['english_1', 'english_2'],
    'Math': ['math_1', 'math_2'],
    'Coding': ['coding_1', 'coding_2', 'coding_3'],
    'Reasoning': ['reasoning_1', 'reasoning_2', 'reasoning_3']
}

In [ ]:
# === Load Dataset ===
df = pd.read_csv("Training/ranked_responses_final.csv")

In [ ]:
# === Embed and Save Per Category ===
for category, sources in category_map.items():
    print(f"\n{'='*10} Processing category: {category} {'='*10}")
    df_cat = df[df["source"].isin(sources)].copy()
    df_cat["clean_prompt"] = df_cat["Prompt"].apply(preprocess_text)
    unique_prompts = df_cat["Prompt"].unique()

    data_list = []
    for prompt in unique_prompts:
        subset = df_cat[df_cat["Prompt"] == prompt]
        models = subset["Model"].tolist()
        ranks = subset["Rank"].tolist()
        if len(ranks) != 4:
            continue
        scores = compute_proportional_ranks(ranks)
        embedding = get_ada_embedding(prompt)
        data_list.append({
            "prompt": prompt,
            "embedding": embedding,
            "scores": dict(zip(models, scores)),
            "source": subset["source"].values[0]
        })

    df_embedded = pd.DataFrame(data_list)
    with open(f"df_embeddings_{category.lower()}.pkl", "wb") as f:
        pickle.dump(df_embedded, f)
    print(f"Saved {len(df_embedded)} prompts to df_embeddings_{category.lower()}.pkl")

In [ ]:
# === Training Per Category using XGBoost ===
class TqdmJoblib:
    def __init__(self, tqdm_object):
        self.tqdm_object = tqdm_object

    def __enter__(self):
        self.original_callback = joblib.parallel.BatchCompletionCallBack
        tqdm_obj = self.tqdm_object

        class TqdmBatchCompletionCallBack(self.original_callback):
            def __call__(self, *args, **kwargs):
                result = super().__call__(*args, **kwargs)
                tqdm_obj.update(n=self.batch_size)
                return result

        joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallBack
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        joblib.parallel.BatchCompletionCallBack = self.original_callback

for category in category_map.keys():
    print(f"\n{'='*20} TRAINING: {category} {'='*20}")
    with open(f"df_embeddings_{category.lower()}.pkl", "rb") as f:
        df_processed = pickle.load(f)

In [ ]:
# Stack embeddings into a numpy array and convert scores into a DataFrame
X = np.vstack(df_processed["embedding"])
y = pd.DataFrame(df_processed["scores"].tolist())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Here we use XGBRegressor wrapped with MultiOutputRegressor for multi-target regression.
model = MultiOutputRegressor(
    XGBRegressor(random_state=42, objective='reg:squarederror', n_jobs=1),
    n_jobs=2
)

In [ ]:
# Grid search over XGBoost hyperparameters
param_grid = {
    "estimator__max_depth": [3, 5, 10],
    "estimator__learning_rate": [0.01, 0.1],
    "estimator__n_estimators": [100]
}

total_combinations = np.prod([len(v) for v in param_grid.values()])
total_iterations = total_combinations * 5  # 5-fold CV

grid_search = GridSearchCV(
    model,
    param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=1,
    verbose=0,
    return_train_score=True
)

with TqdmJoblib(tqdm(total=total_iterations, desc=f"Grid Search: {category}")):
    grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
joblib.dump(best_model, f"model_{category.lower()}.pkl")
print(f"Model saved: model_{category.lower()}.pkl")

In [ ]:
 # Evaluate the model on the test set
y_pred = best_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MSE: {mse:.4f}, MAE: {mae:.4f}, R2: {r2:.4f}")

In [ ]:
# Compute confusion matrix metrics by converting continuous scores to rankings
actual_ranks, predicted_ranks = [], []
model_names = y.columns
for i in range(len(X_test)):
    a_scores = dict(zip(model_names, y_test.iloc[i].values))
    p_scores = dict(zip(model_names, y_pred[i]))
    a_rank = convert_to_ranking(a_scores)
    p_rank = convert_to_ranking(p_scores)
    for m in model_names:
        actual_ranks.append(a_rank[m])
        predicted_ranks.append(p_rank[m])

conf_mat = confusion_matrix(actual_ranks, predicted_ranks, labels=[1, 2, 3, 4])
print(f"Accuracy: {accuracy_score(actual_ranks, predicted_ranks):.4f}")
print(f"Precision: {precision_score(actual_ranks, predicted_ranks, average='macro'):.4f}")
print(f"Recall: {recall_score(actual_ranks, predicted_ranks, average='macro'):.4f}")
print(f"F1 Score: {f1_score(actual_ranks, predicted_ranks, average='macro'):.4f}")

plt.figure(figsize=(6, 5))
sns.heatmap(conf_mat, annot=True, fmt='d', cmap='Blues',
            xticklabels=[1, 2, 3, 4], yticklabels=[1, 2, 3, 4])
plt.title(f"Confusion Matrix - {category}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig(f"confusion_matrix_{category.lower()}.png")
plt.close()
print(f"Confusion matrix saved: confusion_matrix_{category.lower()}.png")